# Phase 1 演習 — CAE結果を読むための材料力学

対応する本文：[`docs/texts/phase1-stress-strain.md`](../docs/texts/phase1-stress-strain.md)

このNotebookは提出用です。各 `あなたの回答` セルを埋め、コードセルを実行してください。数値だけでなく、**どの仮定の下で何を意味する値か**を記述します。

## 提出情報

- 氏名・日付：
- 実行環境（任意）：
- 参照した資料・CAE結果（任意）：

In [1]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)

---

## 演習 1 — コンターの値を説明する

ブラケット根元に高い von Mises 応力のコンターが表示されたとします。次の二点を、それぞれ3〜6文程度で説明してください。

1. この表示値が得られるまでの計算の連鎖を、外力・境界条件から書く。
2. このピークをただちに降伏判定へ使えない場合を二つ挙げ、追加で確認すべきものを述べる。

### あなたの回答（演習 1）

1. 外力・境界条件および形状から平衡状態の節点変位が計算される。変位の勾配からひずみが計算される。計算されたひずみと構成則から応力が計算される。各積分点で計算された応力テンソルの固有値、すなわち主応力からvon Mises応力が計算され、各節点に外挿あるいは平均化されて表示される。
2. ブラケット根元の節点が完全拘束されていた場合、ひずみおよび応力が発散し、数値計算上発生する特異点になっている可能性がある。また、von Mises応力は実際に発生する応力ではなく、テンソルをスカラーに押し込んだ指標であるため、材料や形状、荷重条件によっては表示されたvon Mises応力が発生する以前に破断する可能性がある。したがって、拘束条件の妥当性や荷重伝達経路、変形の様子を追加で確認すべきである。

---

## 演習 2 — 変位からひずみを作る

長さ $L=100\ \mathrm{mm}$ の1D棒を考えます。左端は固定、右端の変位は $u(L)=0.20\ \mathrm{mm}$ です。棒の内部で変位が線形に変化すると仮定します。

1. $u(x)$ を求める。
2. 軸ひずみ $\varepsilon_{xx}=du/dx$ を求める。
3. 右端変位を2倍にした場合、ひずみがどう変わるかを説明する。

単位をそろえれば、ひずみは無次元です。

### あなたの回答（演習 2-1〜3）

1. 変位が線形に変化することから、
   ```math
    u(x) = \frac{u(L)}{L}x = \frac{0.20}{100}x = 0.0020x
   ```

2. ```math
    \varepsilon_{xx} = \frac{du}{dx} = 0.0020
   ```

3. 右端変位を2倍にした場合、$u(x) = 0.0040x$ および $\varepsilon_{xx} = 0.0040$ より、軸ひずみも2倍になる。

In [2]:
# 演習 2-4: 上の導出を数値で確認する。
L_mm = 100.0
u_right_mm = 0.20

# TODO: eps_xx に軸ひずみを代入する。
eps_xx = u_right_mm / L_mm

# TODO: x_mm を5点取り、変位 u_mm を計算する。
x_mm = np.linspace(0.0, L_mm, 5)
u_mm = eps_xx * x_mm

print('x [mm] =', x_mm)
print('u [mm] =', u_mm)
print('epsilon_xx =', eps_xx)

x [mm] = [  0.  25.  50.  75. 100.]
u [mm] = [0.   0.05 0.1  0.15 0.2 ]
epsilon_xx = 0.002


---

## 演習 3 — ひずみから応力を得る

等方・均質・微小ひずみ・線形弾性を仮定し、鋼として $E=210\ \mathrm{GPa}$、$\nu=0.30$ を用います。

1. 演習2の変形が、側面自由の一軸応力状態にあるとき、$\sigma_{xx}$ を求める。
2. 横ひずみ $\varepsilon_{yy}=\varepsilon_{zz}$ を求める。
3. 横ひずみをゼロに拘束する一軸ひずみ状態では、なぜ応力状態が変わるかを説明する。

ここでは $\sigma=E\varepsilon$ を使える条件と、使えない条件を区別することが目的です。

### あなたの回答（演習 3-1〜3）

1. 側面自由の一軸応力状態であることから、
   ```math
    \sigma_{xx} = E \varepsilon_{xx} = 210 \ \mathrm{GPa} \times 0.0020 = 420 \ \mathrm{MPa}
   ```
2. ```math
    \varepsilon_{yy} = \varepsilon_{zz} = -\nu \varepsilon_{xx} = -0.30 \times 0.0020 = -0.00060
   ```
3. 等方・均質・微小ひずみ・線形弾性の条件から、$\boldsymbol{\sigma} = \lambda \mathrm{tr}(\boldsymbol{\varepsilon}) \mathbf{I} + 2 \mu \boldsymbol{\varepsilon}$ であるが、側面自由のとき、$\mathrm{tr}(\boldsymbol{\varepsilon}) = 0.0020 - 0.00060 - 0.00060 = 0.00080$ であるのに対し、横ひずみをゼロに拘束すると、$\mathrm{tr}(\boldsymbol{\varepsilon}) = 0.0020 + 0 + 0 = 0.0020$ となるため、$\sigma_{xx}$ は増加する。実際、横ひずみがゼロのとき、
   ```math
    \sigma_{xx} = (\lambda + 2 \mu) \varepsilon_{xx}
   ```
   ここで,
   ```math
    \lambda = \frac{E \nu}{(1 + \nu) (1 - 2 \nu)}, \ \mu = \frac{E}{2 (1 + \nu)}
   ```
   であるから、
   ```math
    \sigma_{xx} = \frac{E (1 - \nu)}{(1 + \nu)(1 - 2 \nu)} \varepsilon_{xx} = 565 \ \mathrm{MPa}
   ```
   と計算され、3-1 で求めた $420 \ \mathrm{MPa}$ より大きな値となる。
  


In [5]:
def strain_to_voigt(eps):
    """対称ひずみテンソルを工学せん断ひずみを使うVoigt表記へ変換する。"""
    return np.array([
        eps[0, 0], eps[1, 1], eps[2, 2],
        2.0 * eps[1, 2], 2.0 * eps[0, 2], 2.0 * eps[0, 1],
    ])

def voigt_to_tensor(v):
    """工学せん断ひずみを使うVoigt表記を対称テンソルへ戻す。"""
    return np.array([
        [v[0], v[5] / 2.0, v[4] / 2.0],
        [v[5] / 2.0, v[1], v[3] / 2.0],
        [v[4] / 2.0, v[3] / 2.0, v[2]],
    ])

def elasticity_matrix_3d(E, nu):
    """3D等方線形弾性の構成行列。応力の単位はEと同じ。"""
    coef = E / ((1.0 + nu) * (1.0 - 2.0 * nu))
    return coef * np.array([
        [1.0 - nu, nu, nu, 0.0, 0.0, 0.0],
        [nu, 1.0 - nu, nu, 0.0, 0.0, 0.0],
        [nu, nu, 1.0 - nu, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, (1.0 - 2.0 * nu) / 2.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, (1.0 - 2.0 * nu) / 2.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, (1.0 - 2.0 * nu) / 2.0],
    ])

def stress_from_strain(eps, E, nu):
    sigma_v = elasticity_matrix_3d(E, nu) @ strain_to_voigt(eps)
    return voigt_to_tensor(sigma_v)

In [6]:
# 演習 3-4: 一軸応力状態をテンソルと構成則で再現する。
E = 210e9
nu = 0.30

# TODO: 演習2で求めた eps_xx を使い、側面自由のひずみテンソル eps_uniaxial を作る。
# ヒント: epsilon_yy = epsilon_zz = -nu * epsilon_xx
eps_uniaxial = np.array([
    [eps_xx, 0.0, 0.0],
    [0.0, -nu * eps_xx, 0.0],
    [0.0, 0.0, -nu * eps_xx],
])

# TODO: sigma_uniaxial を計算し、sigma_yy と sigma_zz がゼロに近いか確認する。
sigma_uniaxial = stress_from_strain(eps_uniaxial, E, nu)
print(sigma_uniaxial / 1e6, 'MPa')

[[420.   0.   0.]
 [  0.   0.   0.]
 [  0.   0.   0.]] MPa


---

## 演習 4 — 主応力は何をしているか

次の応力テンソル（単位：MPa）を考えます。

$$\boldsymbol{\sigma}=\begin{bmatrix}120 & 50 & 0\\50 & 20 & 0\\0 & 0 & 10\end{bmatrix}$$

1. 固有値・固有ベクトルを求める。
2. 固有ベクトルを列に並べた行列を $\mathbf{Q}$ として、$\mathbf{Q}^T\boldsymbol{\sigma}\mathbf{Q}$ を計算する。
3. 非対角成分が何を意味するか、主応力が「新しい応力」ではない理由とともに説明する。

### あなたの回答（演習 4-3）

非対角成分はすべてゼロであり、これはせん断応力がゼロであることを示している。主応力は主軸方向すなわち応力テンソルの固有ベクトルを基底とする座標系で応力状態を記述したときの引張・圧縮応力であり、このときせん断応力は考えなくてよい。

In [7]:
sigma_mpa = np.array([[120.0, 50.0, 0.0],
                      [ 50.0, 20.0, 0.0],
                      [  0.0,  0.0, 10.0]])

principal_stresses, Q = np.linalg.eigh(sigma_mpa)
sigma_principal = Q.T @ sigma_mpa @ Q

print('principal stresses [MPa] =', principal_stresses)
print('Q^T sigma Q [MPa] =\n', sigma_principal)
assert np.allclose(sigma_principal, np.diag(principal_stresses))

principal stresses [MPa] = [ -0.710678  10.       140.710678]
Q^T sigma Q [MPa] =
 [[ -0.710678   0.         0.      ]
 [  0.        10.         0.      ]
 [ -0.         0.       140.710678]]


---

## 演習 5 — 結果レビュー

手元のCAE結果、または想定したブラケット解析を一つ選び、次を埋めてください。固有名詞や機密値は抽象化して構いません。

- 見た物理量と単位：
- 値の評価位置（積分点、要素中心、節点平均など）：
- 荷重導入部から拘束部までの荷重伝達経路：
- 最大値の近傍にある、モデル化上の注意点：
- 追加で行う検証（反力釣合い、メッシュ収束、出力位置比較など）：
- この値を意思決定に使う前に確認すべき仮定：

### あなたの回答（演習 5）

- 見た物理量と単位：von Mises応力(MPa)
- 値の評価位置（積分点、要素中心、節点平均など）：節点平均
- 荷重導入部から拘束部までの荷重伝達経路：ブラケット端点から剛体との接合部にかけて
- 最大値の近傍にある、モデル化上の注意点：要素長を小さくとる
- 追加で行う検証（反力釣合い、メッシュ収束、出力位置比較など）：反力釣合い
- この値を意思決定に使う前に確認すべき仮定：要素長が大きく接合部が特異点となっている

## 振り返り

1. 今回、最も理解が変わった概念：
2. まだ説明できない／導出したい概念：
3. 教材または演習の改善提案：

この内容は `docs/learning-log.md` のPhase 1項目へ追記してください。